# 🔍 Clase 3 — Valores Faltantes: Detección, Diagnóstico e Imputación (Python)

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · drdarioezequieldiaz@gmail.com

**Fecha:** Jueves 23 de abril de 2026

---

### Objetivos del notebook

1. **Simular** los tres mecanismos de Rubin (MCAR, MAR, MNAR) sobre un dataset financiero real.
2. **Visualizar** patrones de ausencia con `missingno` y matrices de correlación de indicadores.
3. **Diagnosticar** el mecanismo mediante test de Little, regresión logística del indicador de ausencia y correlación point-biserial.
4. **Implementar** cinco estrategias de imputación: eliminación, media/mediana, KNN, imputación iterativa (MICE).
5. **Comparar** el impacto de cada estrategia sobre la distribución y la capacidad predictiva.
6. **Demostrar** empíricamente el sesgo que introduce la listwise deletion bajo MAR/MNAR.

### Estrategia pedagógica

Partimos del **German Credit Dataset completo** (0% missing). Introducimos valores faltantes artificialmente bajo cada mecanismo de Rubin en la variable `credit_amount`, lo que nos permite **comparar cada imputación contra el valor real** — algo imposible con datos faltantes genuinos.

> **Referencia teórica:** Apunte de Clase 2, Parte I — *Valores Faltantes en Bases de Datos Financieras* (Díaz, 2026).


---
## 1. Configuración del entorno


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. CONFIGURACIÓN DEL ENTORNO
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi'] = 100

# Paleta del curso
COL_GOOD, COL_BAD, COL_ACC = '#2a9d8f', '#e76f51', '#f4a261'

print("✅ Entorno configurado correctamente")

# Verificar missingno
try:
    import missingno as msno
    print("✅ missingno disponible")
except ImportError:
    print("📦 Instalando missingno...")
    !pip install missingno -q
    import missingno as msno
    print("✅ missingno instalado")


---
## 2. Carga del dataset y línea base

Cargamos el German Credit Dataset completo. Al no tener valores faltantes, nos sirve como **ground truth** para evaluar la calidad de cada imputación.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. CARGA DEL DATASET
# ══════════════════════════════════════════════════════════════

credit = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
df_original = credit.frame.copy()

# Seleccionamos variables numéricas clave para el análisis
num_vars = ['duration', 'credit_amount', 'installment_commitment',
            'residence_since', 'age', 'existing_credits', 'num_dependents']

print(f"✅ Dataset cargado: {df_original.shape[0]} filas × {df_original.shape[1]} columnas")
print(f"   Valores faltantes totales: {df_original.isnull().sum().sum()}")
print(f"\n   Estadísticos de credit_amount (ground truth):")
print(f"   Media  = {df_original['credit_amount'].mean():.2f}")
print(f"   Mediana = {df_original['credit_amount'].median():.2f}")
print(f"   Desvío = {df_original['credit_amount'].std():.2f}")


### 📝 Interpretación

El dataset está completo: 0 valores faltantes. La variable `credit_amount` (monto del crédito) tiene media ≈ 3271, mediana ≈ 2320 y desvío ≈ 2823. La diferencia media-mediana evidencia la asimetría positiva típica de variables monetarias. Estos valores constituyen nuestra **línea base** contra la cual evaluaremos cada estrategia de imputación.

---
## 3. Simulación de los tres mecanismos de Rubin

Introducimos un 20% de valores faltantes en `credit_amount` bajo cada mecanismo, para demostrar empíricamente cómo la naturaleza de la ausencia condiciona los resultados.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. SIMULACIÓN DE MECANISMOS DE RUBIN
# ══════════════════════════════════════════════════════════════

np.random.seed(42)
n = len(df_original)
prop_missing = 0.20  # 20% de faltantes

# ── 3.1 MCAR: ausencia puramente aleatoria ──
mask_mcar = np.random.rand(n) < prop_missing

# ── 3.2 MAR: ausencia depende de 'age' (observada) ──
# Jóvenes tienen mayor probabilidad de no reportar el monto
prob_mar = 1 / (1 + np.exp(0.08 * (df_original['age'].astype(float) - 28)))
prob_mar = prob_mar / prob_mar.mean() * prop_missing  # escalar a ~20% global
prob_mar = prob_mar.clip(0.01, 0.95)
mask_mar = np.random.rand(n) < prob_mar

# ── 3.3 MNAR: ausencia depende del propio credit_amount ──
# Montos altos tienen mayor probabilidad de no ser reportados
ca = df_original['credit_amount'].astype(float)
prob_mnar = 1 / (1 + np.exp(-0.0008 * (ca - ca.quantile(0.7))))
prob_mnar = prob_mnar / prob_mnar.mean() * prop_missing
prob_mnar = prob_mnar.clip(0.01, 0.95)
mask_mnar = np.random.rand(n) < prob_mnar

# Crear DataFrames con faltantes
df_mcar = df_original.copy(); df_mcar.loc[mask_mcar, 'credit_amount'] = np.nan
df_mar  = df_original.copy(); df_mar.loc[mask_mar, 'credit_amount'] = np.nan
df_mnar = df_original.copy(); df_mnar.loc[mask_mnar, 'credit_amount'] = np.nan

print("=" * 65)
print("RESUMEN DE FALTANTES SIMULADOS EN credit_amount")
print("=" * 65)
for name, mask in [("MCAR", mask_mcar), ("MAR", mask_mar), ("MNAR", mask_mnar)]:
    n_miss = mask.sum()
    mean_obs = df_original.loc[~mask, 'credit_amount'].astype(float).mean()
    mean_mis = df_original.loc[mask, 'credit_amount'].astype(float).mean()
    print(f"\n  {name}:")
    print(f"    Faltantes: {n_miss} ({n_miss/n*100:.1f}%)")
    print(f"    Media de los OBSERVADOS: {mean_obs:,.0f}")
    print(f"    Media de los FALTANTES (ground truth): {mean_mis:,.0f}")
    print(f"    Diferencia: {abs(mean_obs - mean_mis):,.0f}" +
          (" ← sesgo!" if abs(mean_obs - mean_mis) > 200 else " ← sin sesgo"))


### 📝 Interpretación

Este resultado constituye la **demostración empírica del Teorema 4.1** del apunte teórico:

- **MCAR:** La media de los valores observados y la de los faltantes son prácticamente iguales. No hay sesgo: la ausencia es independiente del valor.
- **MAR:** Existe una diferencia moderada porque los jóvenes (que solicitan montos distintos) tienen mayor probabilidad de faltante.
- **MNAR:** La diferencia es sustancial porque los montos altos tienen mayor probabilidad de faltar. La media de los observados **subestima** la media real.

---
## 4. Visualización de patrones de ausencia


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. VISUALIZACIÓN CON missingno
# ══════════════════════════════════════════════════════════════

# Creamos un DataFrame con múltiples variables faltantes para visualizar
df_demo = df_original[num_vars].copy()
df_demo['credit_amount'] = df_mcar['credit_amount']  # MCAR
# Agregar faltantes en 'age' bajo MAR (depende de duration)
mask_age = np.random.rand(n) < (0.15 * (df_original['duration'].astype(float) > 24).astype(int) + 0.05)
df_demo.loc[mask_age, 'age'] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matriz de ausencia
msno.matrix(df_demo, ax=axes[0], sparkline=False, fontsize=10, color=(0.16, 0.62, 0.54))
axes[0].set_title('Matriz de ausencia (missingno)', fontweight='bold', fontsize=12)

# Heatmap de correlación entre indicadores de ausencia
missing_corr = df_demo.isnull().corr()
sns.heatmap(missing_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1], vmin=-1, vmax=1, square=True, linewidths=0.5)
axes[1].set_title('Correlación entre indicadores de ausencia', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('fig_01_patron_ausencia.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Figura guardada: fig_01_patron_ausencia.png")


### 📝 Interpretación

- **Matriz de ausencia (izquierda):** Las barras blancas señalan los valores faltantes. Si los patrones fueran aleatorios (MCAR), las bandas blancas se distribuirían uniformemente. Bloques alineados sugieren MAR o MNAR.
- **Correlación de indicadores (derecha):** Mide si la ausencia en una variable correlaciona con la ausencia en otra. Correlaciones altas sugieren un mecanismo compartido (por ejemplo, el mismo formulario omite ambos campos).

---
## 5. Diagnóstico del mecanismo de ausencia

### 5.1 Regresión logística del indicador de ausencia

Para cada mecanismo, modelamos $P(R = 0 | \mathbf{X})$ como función de las variables observadas. Si el AUC > 0.5 significativamente, la ausencia depende de variables observadas → MAR o MNAR.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5.1 REGRESIÓN LOGÍSTICA DEL INDICADOR DE AUSENCIA
# ══════════════════════════════════════════════════════════════

print("=" * 65)
print("DIAGNÓSTICO: REGRESIÓN LOGÍSTICA DE R (indicador de ausencia)")
print("=" * 65)

# Variables predictoras para el diagnóstico (excluimos credit_amount)
X_diag = df_original[['duration', 'installment_commitment', 'residence_since',
                        'age', 'existing_credits', 'num_dependents']].astype(float)

for name, mask in [("MCAR", mask_mcar), ("MAR", mask_mar), ("MNAR", mask_mnar)]:
    y_missing = mask.astype(int)  # 1 = faltante

    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_diag, y_missing)
    probs = lr.predict_proba(X_diag)[:, 1]
    auc = roc_auc_score(y_missing, probs)

    print(f"\n  {name}:")
    print(f"    AUC-ROC = {auc:.4f}", end="")
    if auc < 0.55:
        print("  →  Ausencia NO predecible → compatible con MCAR ✅")
    elif auc < 0.65:
        print("  →  Predicción débil → posible MAR leve")
    else:
        print("  →  Ausencia PREDECIBLE por variables observadas → MAR ⚠️")

    # Coeficientes más relevantes
    coefs = pd.Series(lr.coef_[0], index=X_diag.columns).abs().sort_values(ascending=False)
    top_var = coefs.index[0]
    print(f"    Variable más predictiva de la ausencia: '{top_var}' (|β| = {coefs.iloc[0]:.4f})")


### 📝 Interpretación

- **MCAR:** AUC ≈ 0.50 — las variables observadas NO predicen la ausencia. Coherente con el mecanismo puramente aleatorio.
- **MAR:** AUC significativamente > 0.50 — la edad predice la ausencia. La variable `age` domina los coeficientes.
- **MNAR:** AUC moderado — las variables observadas capturan parte de la señal (porque `credit_amount` correlaciona con `duration` y `age`), pero no toda, ya que la dependencia principal es con el valor faltante mismo.

### 5.2 Correlación point-biserial entre R y variables observadas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5.2 CORRELACIÓN POINT-BISERIAL
# ══════════════════════════════════════════════════════════════

print("=" * 65)
print("CORRELACIÓN POINT-BISERIAL: R(credit_amount) vs. variables")
print("=" * 65)

for name, mask in [("MCAR", mask_mcar), ("MAR", mask_mar), ("MNAR", mask_mnar)]:
    print(f"\n  {name}:")
    r_indicator = mask.astype(int)
    for col in ['duration', 'age', 'installment_commitment']:
        r_pb, p_val = stats.pointbiserialr(r_indicator, df_original[col].astype(float))
        sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns"))
        print(f"    vs. {col:<28s} r_pb = {r_pb:+.4f}  (p = {p_val:.4f}) {sig}")


### 📝 Interpretación

- **MCAR:** Ninguna correlación significativa — la ausencia es independiente de todas las variables.
- **MAR:** Correlación fuerte con `age` — confirma que la edad condiciona la ausencia.
- **MNAR:** Patrón mixto — las correlaciones observadas reflejan la estructura indirecta, no la dependencia principal (que es con el valor faltante).

---
## 6. Estrategias de imputación: implementación y comparación


In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. IMPUTACIÓN — 5 ESTRATEGIAS
# ══════════════════════════════════════════════════════════════

def evaluate_imputation(df_with_na, df_true, mask, method_name, imputed_values):
    """Evalúa la calidad de una imputación comparando con ground truth."""
    true_vals = df_true.loc[mask, 'credit_amount'].astype(float).values
    imp_vals = imputed_values[mask]

    mae = np.mean(np.abs(true_vals - imp_vals))
    rmse = np.sqrt(np.mean((true_vals - imp_vals)**2))
    mean_imp = np.mean(imp_vals)
    mean_true = np.mean(true_vals)
    bias = mean_imp - mean_true

    return {
        'Método': method_name,
        'MAE': round(mae, 1),
        'RMSE': round(rmse, 1),
        'Media imputada': round(mean_imp, 1),
        'Media real': round(mean_true, 1),
        'Sesgo': round(bias, 1),
        'Sesgo %': round(bias / mean_true * 100, 1)
    }

# Preparar datos numéricos para imputación
results_all = {}

for mech_name, df_na, mask in [("MCAR", df_mcar, mask_mcar),
                                 ("MAR", df_mar, mask_mar),
                                 ("MNAR", df_mnar, mask_mnar)]:
    X = df_na[num_vars].astype(float).copy()
    results = []

    # M1: Listwise deletion (media de los completos)
    mean_cc = X['credit_amount'].dropna().mean()
    imp_lw = X['credit_amount'].copy()
    imp_lw[mask] = mean_cc  # "imputamos" con la media de completos para comparar
    results.append(evaluate_imputation(df_na, df_original, mask,
                                        "Listwise (media CC)", imp_lw.values))

    # M2: Imputación por mediana
    imp_median = SimpleImputer(strategy='median')
    X_median = pd.DataFrame(imp_median.fit_transform(X), columns=num_vars)
    results.append(evaluate_imputation(df_na, df_original, mask,
                                        "Mediana", X_median['credit_amount'].values))

    # M3: Imputación por media
    imp_mean = SimpleImputer(strategy='mean')
    X_mean = pd.DataFrame(imp_mean.fit_transform(X), columns=num_vars)
    results.append(evaluate_imputation(df_na, df_original, mask,
                                        "Media", X_mean['credit_amount'].values))

    # M4: KNN Imputer (k=5)
    imp_knn = KNNImputer(n_neighbors=5)
    X_knn = pd.DataFrame(imp_knn.fit_transform(X), columns=num_vars)
    results.append(evaluate_imputation(df_na, df_original, mask,
                                        "KNN (k=5)", X_knn['credit_amount'].values))

    # M5: Iterative Imputer (MICE-like)
    imp_mice = IterativeImputer(max_iter=10, random_state=42)
    X_mice = pd.DataFrame(imp_mice.fit_transform(X), columns=num_vars)
    results.append(evaluate_imputation(df_na, df_original, mask,
                                        "MICE (iterativo)", X_mice['credit_amount'].values))

    results_all[mech_name] = pd.DataFrame(results)

# Mostrar resultados por mecanismo
for mech_name, res_df in results_all.items():
    print(f"\n{'='*70}")
    print(f"  COMPARACIÓN DE IMPUTACIONES — Mecanismo: {mech_name}")
    print(f"{'='*70}")
    print(res_df.to_string(index=False))


### 📝 Interpretación

Los resultados confirman empíricamente los teoremas del apunte teórico:

**Bajo MCAR:**
- Todas las estrategias producen estimaciones razonables con sesgo bajo.
- La listwise deletion es insesgada (como demuestra el Teorema 3.1 del apunte).
- KNN y MICE ofrecen menor MAE/RMSE porque aprovechan la estructura multivariada.

**Bajo MAR:**
- La listwise deletion introduce **sesgo significativo** (la media de los complete cases difiere de la media real — Teorema 4.1 del apunte).
- La imputación por media/mediana también es sesgada porque no condiciona en la edad.
- **KNN y MICE reducen el sesgo** porque capturan la relación entre `credit_amount` y `age`.

**Bajo MNAR:**
- **Todos los métodos están sesgados** (Proposición 5.1 del apunte).
- KNN y MICE mejoran respecto de la media/mediana, pero no eliminan el sesgo.
- La corrección bajo MNAR requiere modelos de selección (Heckman) que exceden el alcance de estos métodos estándar.

---
## 7. Visualización comparativa de distribuciones post-imputación


In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. DISTRIBUCIONES POST-IMPUTACIÓN
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
mechanisms = [("MCAR", df_mcar, mask_mcar),
              ("MAR", df_mar, mask_mar),
              ("MNAR", df_mnar, mask_mnar)]

for ax, (mech_name, df_na, mask) in zip(axes, mechanisms):
    X = df_na[num_vars].astype(float).copy()

    # Ground truth
    true_vals = df_original['credit_amount'].astype(float)
    ax.hist(true_vals, bins=40, density=True, alpha=0.3, color='gray',
            label='Ground truth', edgecolor='white')

    # KNN
    imp_knn = KNNImputer(n_neighbors=5)
    X_knn = pd.DataFrame(imp_knn.fit_transform(X), columns=num_vars)
    ax.hist(X_knn['credit_amount'], bins=40, density=True, alpha=0.5,
            color=COL_GOOD, label='KNN (k=5)', edgecolor='white')

    # Media
    imp_mean = SimpleImputer(strategy='mean')
    X_mean = pd.DataFrame(imp_mean.fit_transform(X), columns=num_vars)
    ax.hist(X_mean['credit_amount'], bins=40, density=True, alpha=0.5,
            color=COL_BAD, label='Media', edgecolor='white')

    ax.set_title(f'{mech_name}', fontweight='bold', fontsize=13)
    ax.set_xlabel('credit_amount')
    ax.legend(fontsize=8)

plt.suptitle('Distribución post-imputación vs. ground truth por mecanismo',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig_02_distribuciones_imputacion.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Figura guardada: fig_02_distribuciones_imputacion.png")


### 📝 Interpretación

Los histogramas superpuestos revelan visualmente el sesgo:

- **MCAR:** Tanto KNN como la media producen distribuciones similares al ground truth.
- **MAR:** La imputación por media genera un pico artificial (spike) en la media global, distorsionando la distribución. KNN preserva mejor la forma original.
- **MNAR:** Ambas estrategias subestiman la cola derecha (montos altos), porque los valores faltantes provienen precisamente de esa zona. El KNN mitiga parcialmente el sesgo pero no lo elimina.

---
## 8. Impacto sobre un modelo de scoring

¿Cómo afecta cada estrategia de imputación al rendimiento de un modelo predictivo?


In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. IMPACTO SOBRE SCORING: AUC POR ESTRATEGIA DE IMPUTACIÓN
# ══════════════════════════════════════════════════════════════

from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

print("=" * 70)
print("IMPACTO EN SCORING: AUC-ROC medio (5-fold CV) por estrategia")
print("=" * 70)

le = LabelEncoder()
y = le.fit_transform(df_original['class'])  # good=1, bad=0

# AUC con datos completos (benchmark)
X_full = df_original[num_vars].astype(float)
lr_full = LogisticRegression(max_iter=1000, random_state=42)
auc_full = cross_val_score(lr_full, X_full, y, cv=5, scoring='roc_auc').mean()
print(f"\n  Datos completos (benchmark): AUC = {auc_full:.4f}")

for mech_name, df_na, mask in mechanisms:
    X_na = df_na[num_vars].astype(float).copy()
    print(f"\n  ── {mech_name} ──")

    strategies = {
        'Listwise deletion': X_na.dropna(),
        'Media': pd.DataFrame(SimpleImputer(strategy='mean').fit_transform(X_na), columns=num_vars),
        'KNN (k=5)': pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(X_na), columns=num_vars),
        'MICE': pd.DataFrame(IterativeImputer(max_iter=10, random_state=42).fit_transform(X_na), columns=num_vars),
    }

    for strat_name, X_imp in strategies.items():
        if strat_name == 'Listwise deletion':
            y_sub = y[X_na['credit_amount'].notna()]
        else:
            y_sub = y

        lr = LogisticRegression(max_iter=1000, random_state=42)
        auc = cross_val_score(lr, X_imp, y_sub, cv=5, scoring='roc_auc').mean()
        delta = auc - auc_full
        print(f"    {strat_name:<22s} AUC = {auc:.4f}  (Δ = {delta:+.4f})")


### 📝 Interpretación

El análisis de impacto en scoring confirma que:

- **MCAR:** Todas las estrategias preservan el AUC razonablemente bien. La listwise deletion pierde ~20% de los datos pero el AUC se mantiene porque no hay sesgo de selección.
- **MAR:** La listwise deletion degrada el AUC más que los métodos de imputación, porque el subconjunto completo ya no es representativo. KNN y MICE preservan mejor el AUC.
- **MNAR:** Ninguna estrategia recupera completamente el AUC del benchmark. El sesgo inherente al mecanismo MNAR impone un **techo de rendimiento** que solo modelos de selección (Heckman) podrían superar.

---
## 9. Resumen ejecutivo y buenas prácticas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. RESUMEN EJECUTIVO
# ══════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════╗")
print("║                    RESUMEN EJECUTIVO                           ║")
print("║       Valores Faltantes — Detección e Imputación               ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║                                                                ║")
print("║  PROTOCOLO RECOMENDADO:                                        ║")
print("║  1. Cuantificar: % faltantes por variable y por registro       ║")
print("║  2. Visualizar: missingno.matrix() para detectar patrones      ║")
print("║  3. Diagnosticar: regresión logística de R → AUC > 0.55 = MAR ║")
print("║  4. Imputar: KNN o MICE como primera opción bajo MAR          ║")
print("║  5. Validar: comparar distribuciones pre/post imputación       ║")
print("║  6. Pipeline: fit en train, transform en test (NUNCA global)   ║")
print("║                                                                ║")
print("║  RESULTADO CLAVE:                                              ║")
print("║  · MCAR → listwise deletion válida (pero ineficiente)         ║")
print("║  · MAR  → KNN/MICE insesgados; listwise/media sesgados        ║")
print("║  · MNAR → todos sesgados; requiere modelos de selección        ║")
print("╚══════════════════════════════════════════════════════════════════╝")


---
## 📚 Referencias

- **Rubin, D. B.** (1976). Inference and Missing Data. *Biometrika*, 63(3), 581–592.
- **Little, R. J. A.** (1988). A Test of MCAR. *JASA*, 83(404), 1198–1202.
- **Van Buuren, S.** (2018). *Flexible Imputation of Missing Data*. CRC Press.
- **Díaz, D. E.** (2026). Apunte Clase 2, Parte I: *Valores Faltantes en Bases de Datos Financieras*. UTN FRRo.

---
> **Dr. Darío Ezequiel Díaz** · Aplicaciones de Minería de Datos a Economía y Finanzas · MMD UTN FRRo · 2026
